#### **Testing the Full GPT-2 + RAG Pipeline**

In [ ]:
import torch, tiktoken

# your own GPT-2 architecture + config class
from GPT2_Pretrained import GPT 

# "rag." prefix needed since this script sits outside the rag/ package
from rag.retriever import HybridRetriever
from rag.pipeline import answer_question
from rag.chunking import load_documents, chunk_documents 

# load every .txt in ./data -> list of {"id", "text"} dicts
docs = load_documents("./data")

# split into ~150-token overlapping chunks -> list of {"chunk_id", "doc_id", "text"}
chunks = chunk_documents(docs)

# builds BOTH BM25 (sparse) and FAISS (dense) indexes -- the slow, embed-everything
# step, so it's done ONCE here, not inside answer_question on every call
retriever = HybridRetriever(chunks)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# same tokenizer used for chunking and the model itself, so token counts line up
enc = tiktoken.get_encoding('gpt2')

# stop-generation signal + padding token
EOS_ID = enc.eot_token

# map_location handles device mismatches (e.g. trained on GPU, running on CPU)
# weights_only=False since the checkpoint also holds optimizer state + config, not just weights
ckpt = torch.load('checkpoints/gpt2_124M_squad_ft.pt', map_location=device, weights_only=False)

# rebuild the exact architecture the checkpoint was trained with
model = GPT(ckpt['config'])
model.load_state_dict(ckpt['model'])
model.to(device); model.eval()  # eval mode disables dropout etc.

# retrieve top-k chunks -> build context -> prompt -> generate -> return all three for debugging
result = answer_question(
    retriever, model, enc, device, EOS_ID, 1024,
    question="How many championships has Faker won?",
    method="hybrid",
)

# print results and the context used
print("Answer:", result["answer"])
print("\nContext used:\n", result["context_used"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Answer: six

Context used:
  player in the history of the game.

Outside of Worlds, Faker has won the Mid-Season Invitational twice, in 2016 and 2017, along with a large number of LCK titles in the Korean domestic league. He is also known for a wide champion pool. Over the years he has been associated with LeBlanc, Ryze, Azir, Syndra, Orianna and many others, and he was among the first mid laners to show how flexible a single role could be.

His playstyle is defined by mechanical precision, calm decision making and strong map awareness. Teammates and coaches often say his greatest strength is not his hands but his reading of the game: he seems to know where the enemy jungler
 reinventing his role. He shifted from a pure carry into a strategic leader who set the tempo of the whole team, and he kept adapting to new champions and new patches every season.

In 2023 he lifted his fourth World Championship trophy, with T1 beating Weibo Gaming in Seoul, seven years after his previous one. In 

In [ ]:
# retrieve top-k chunks -> build context -> prompt -> generate -> return all three for debugging
result = answer_question(
    retriever, model, enc, device, EOS_ID, 1024, question="How many worlds titles does faker have? ", method="hybrid",)

print("Answer:", result["answer"])
print("\nContext used:\n", result["context_used"])

Answer: six

Context used:
  player in the history of the game.

Outside of Worlds, Faker has won the Mid-Season Invitational twice, in 2016 and 2017, along with a large number of LCK titles in the Korean domestic league. He is also known for a wide champion pool. Over the years he has been associated with LeBlanc, Ryze, Azir, Syndra, Orianna and many others, and he was among the first mid laners to show how flexible a single role could be.

His playstyle is defined by mechanical precision, calm decision making and strong map awareness. Teammates and coaches often say his greatest strength is not his hands but his reading of the game: he seems to know where the enemy jungler
 reinventing his role. He shifted from a pure carry into a strategic leader who set the tempo of the whole team, and he kept adapting to new champions and new patches every season.

In 2023 he lifted his fourth World Championship trophy, with T1 beating Weibo Gaming in Seoul, seven years after his previous one. In 